# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [5]:
# 1. Create a 'decade' column as specified by the formula
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# 2. Filter to the most common ratings
target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_heatmap = df[df['rating'].isin(target_ratings)]

# Group and count the data, then pivot it so ratings are on y-axis and decades on x-axis
heatmap_data = df_heatmap.groupby(['rating', 'decade']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='rating', columns='decade', values='count').fillna(0)

# Ensure the y-axis aligns with the most common ratings hierarchy
heatmap_pivot = heatmap_pivot.reindex(target_ratings)

# 3. Build the Heatmap with Plotly Express
fig1 = px.imshow(
    heatmap_pivot,
    text_auto=True,                # Requirement: Values shown in cells
    color_continuous_scale='Blues', # Requirement: Sequential color scale (Blues)
    labels=dict(x="Decade", y="Content Rating", color="Count of Titles"),
    title="<b>TV-MA and TV-14 Ratings Dominate the 2010s Streaming Era</b>" # Requirement: Insight title
)

# Optional styling tweak to make the layout look polished
fig1.update_layout(title_x=0.5)
fig1.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [6]:
# 1. Filter to Movies only
df_movies = df[df['type'] == 'Movie'].copy()

# 2. Filter years between 2015 and 2022 and group by added_year
df_movies = df_movies[(df_movies['added_year'] >= 2015) & (df_movies['added_year'] <= 2022)]
waterfall_data = df_movies.groupby('added_year').size().reset_index(name='additions')

# Extract years and the additions counts into lists
years = waterfall_data['added_year'].astype(str).tolist()
additions = waterfall_data['additions'].tolist()

# 3. Configure lists for the Waterfall format (including the final total bar)
x_labels = years + ['Total']
y_values = additions + [0] # The final bar value is computed automatically by setting measure="total"
measures = ["relative"] * len(years) + ["total"]

# 4. Build the Waterfall chart using Plotly Graph Objects
fig2 = go.Figure(go.Waterfall(
    name="Movie Additions",
    orientation="v",
    measure=measures,
    x=x_labels,
    y=y_values,
    # Requirement: Green for additions, Blue for totals (Subtractions aren't present here)
    increasing=dict(marker=dict(color="green")),
    totals=dict(marker=dict(color="blue")),
    textposition="outside",
    text=[str(v) for v in additions] + ["Total"]
))

# 5. Add Titles and the required Max Addition Annotation
max_idx = additions.index(max(additions))
max_year = years[max_idx]
max_val = additions[max_idx]

fig2.update_layout(
    title="<b>Rapid Expansion Peaks in 2019 Before Post-Pandemic Content Slowdown</b>", # Requirement: Insight title
    showlegend=False,
    annotations=[
        dict(
            x=max_year,
            y=max_val,
            text=f"Peak Additions ({max_val})", # Requirement: Annotation on largest single addition year
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-40
        )
    ]
)

fig2.show()